In [152]:
# 사용자 정보를 기반으로 영화를 추천하는 서비스
# 1번 사용자한테 추천할 영화 5개를 선택
# 1번 사용자가 2번 영화에 별점을 몇점 줄것인지 예측

In [153]:
# ratings_small.csv에 있는 데이터를 가져와서 DataFrame으로 변환
import pandas as pd

df_ratings = pd.read_csv('../../data/csv/ratings_small.csv')

# tmdb_5000_credits.csv에 있는 데이터를 가져와서 DataFrame으로 변환
df_movies = pd.read_csv('../../data/csv/tmdb_5000_credits.csv')

In [154]:
# df_movies에 있는 영화 제목을 df_ratings에 추가
# 14일차 02_영화추천예제를 참고
# credits : movie_id, ratings : movieId
df_movies.columns = ['movieId', 'title', 'cast', 'crew']
df = df_ratings.merge(df_movies[['movieId', 'title']], on='movieId')
df.head()

,userId,movieId,rating,timestamp,title
0,1,2105,4.0,1260759139,American Pie
1,1,2294,2.0,1260759108,Jay and Silent Bob Strike Back
2,2,62,3.0,835355749,2001: A Space Odyssey
3,2,153,4.0,835355441,Lost in Translation
4,2,161,3.0,835355493,Ocean's Eleven


In [155]:
# userId의 값을 0부터 맵핑
# 1. 중복된 userId를 제거
user_ids = df['userId'].unique().tolist()
# 2. [{1 : 0, {2 : 0}]형태로 변환 {userId : 순서}
user_to_index = {x : i for i, x in enumerate(user_ids)}

In [156]:
# movieId의 값을 0부터 맵핑
movie_ids = df['movieId'].unique().tolist()
movie_to_index = {x : i for i, x in enumerate(movie_ids)} 

In [157]:
# df에 movie_ids와 user_ids를 추가
df['user'] = df['userId'].map(user_to_index)
df['movie'] = df['movieId'].map(movie_to_index)

In [158]:
# 사용자 수, 아이템(영화)수를 계산
num_users = len(user_ids)
num_movies = len(movie_ids)

In [159]:
import tensorflow as tf
# 학습에 사용할 데이터 셋을 준비
# 입력은 딕셔너리, 정답은 리스트 형태로
user_data = df['user'].values.astype('int32')
movie_data = df['movie'].values.astype('int32')
ratings = df['rating'].values.astype('float32')
# from_tensor_slices((X, y))
dataset = tf.data.Dataset.from_tensor_slices(({
	'user_in' : user_data,
	'movie_in' : movie_data
	}, ratings))
train_ds = dataset.cache().shuffle(100).batch(32).prefetch(32)

In [160]:
# 모델 설계 : Sequential 사용 못함
from tensorflow.keras import layers, models

# 입력층 : 여러개의 입력 중 사용자 정보와 영화 정보를 어디서 가져와야 하는지 지정
user_input = layers.Input(shape=(1,), name='user_in')
movie_input = layers.Input(shape=(1,), name='movie_in')

# 임베딩
user_emb = layers.Embedding(input_dim=num_users, output_dim=16, name='user_emb')(user_input)
movie_emb = layers.Embedding(input_dim=num_movies, output_dim=16, name='movie_emb')(movie_input)

# 벡터 평탄화
user_vec = layers.Flatten()(user_emb)
movie_vec = layers.Flatten()(movie_emb)

# 두 벡터를 결합
x = layers.Concatenate()([user_vec, movie_vec])

# 은닉층
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)

# 출력층
outputs = layers.Dense(1)(x)

model = models.Model(inputs=[user_input, movie_input], outputs=outputs)

# 모델 설정
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [161]:
history = model.fit(train_ds, epochs=10, verbose=1)

Epoch 1/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 2.3352 - mae: 1.1113
Epoch 2/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8094 - mae: 0.6990
Epoch 3/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7552 - mae: 0.6716
Epoch 4/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7412 - mae: 0.6662
Epoch 5/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7341 - mae: 0.6647
Epoch 6/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7194 - mae: 0.6563
Epoch 7/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7026 - mae: 0.6475
Epoch 8/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6780 - mae: 0.6330
Epoch 9/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6589 - mae: 0.6235
Epoch 10/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6396 - mae: 0.6139


In [162]:
import numpy as np
test_user_id = 1 # 실제 1번 회원
test_movie_id = 62 # '2001 : A Space Odyssey'
# df[df['movie'] == 2]['title'].values[0]

test_user = np.array([user_to_index[test_user_id]])
test_movie = np.array([movie_to_index[test_movie_id]])

prediction = model.predict({
	'user_in' : test_user, 'movie_in' : test_movie}, 
	verbose=1)

print('-----------')
res = prediction[0][0]
res = np.round(res / 0.5) * 0.5 # 예상 평점이 2.8이면 반올림해서 3
def get_movie_title(movie_id):
	return df_movies[df_movies['movieId'] == movie_id]['title'].values[0]


title = get_movie_title(test_movie_id)
print(f'사용자 {test_user_id}의 영화 {title}에 대한 예상 평점 :{res:.2f}점')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
-----------
사용자 1의 영화 2001: A Space Odyssey에 대한 예상 평점 :3.00점


In [184]:
test_user_id = 1

# 1번 사용자에게 추천하는 영화 목록
# 1번 사용자가 본 영화를 제외하는 작업

# 1번 사용자가 본 id들
watched_movie_idxs = df[df['userId'] == test_user_id]['movie'].unique()

# 전체 영화 id들
all_movie_idxs = np.array(list(movie_to_index.values()))

# 1번 사용자가 안본 영화 id들
candidate_movie_idxs = all_movie_idxs[~np.isin(all_movie_idxs, watched_movie_idxs)]

print([test_user_id] * 3) # 리스트 * 3은 리스트가 3번 반복
# np.array([test_user_id]) * 3 # 넘파이 리스트 * 3은 리스트의 값을 3배 => [3]
# [10, 20], [1, 1]
test_user_idxs = np.array([user_to_index[test_user_id]] * len(candidate_movie_idxs))

# 영화별 1번 사용자의 예상 추천점수
predictions = model.predict({
	'user_in' : test_user_idxs, 'movie_in' : candidate_movie_idxs
})

# 정렬 : argsort() 값을 기준으로 오름차순으로 정렬하여 인덱스를 뒤에서 5개 반환
# [::-1] 역순
top_idxs = predictions.flatten().argsort()[-5:][::-1]

# {영화번호 : 0, 영화번호 : 1} => {0 : 영화번호, 1 : 영화번호}. k와 v를 반대로
top_ids = {i:movie for movie, i in movie_to_index.items()}
top_titles = [get_movie_title(top_ids[i]) for i in top_idxs]
for i, idx in enumerate(top_idxs):
	print(f'{i+1}위 : {top_titles[i]}(예상 평점 : {predictions[idx][0]:.2f}점)')

[1, 1, 1]
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
1위 : The Passion of the Christ(예상 평점 : 4.81점)
2위 : Riding Giants(예상 평점 : 4.77점)
3위 : Stop-Loss(예상 평점 : 4.76점)
4위 : Stitches(예상 평점 : 4.74점)
5위 : Orgazmo(예상 평점 : 4.70점)
